# Optimal transport for generative models — OT-GAN, corrected, and what came after

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dan-allouche-qf/optimal-transport-gan/blob/main/OT_GAN.ipynb)

Three acts, one evaluation harness throughout (FID@10k + KID + a domain-aware FID-LeNet; train/test floor 1.45 / 1.76):

1. **2018 — correctness.** Re-implementing OT-GAN (Salimans et al., ICLR 2018) exposed five bugs — the worst trained the critic in the **wrong direction**. Corrected and locked by tests: FID@10k **70.8** vs **177.9** buggy, consistent across all three metrics.
2. **2019 — the loss.** The minibatch energy distance is exactly 2× a Sinkhorn divergence with independent-minibatch self-terms; swapping in the **debiased** same-batch self-terms (Genevay, Peyré & Cuturi 2018; Feydy et al. 2019) at equal budget takes FID **81.2 → 24.4** — and the documented critic collapse disappears.
3. **2024 — the successor.** OT-CFM (Tong et al., TMLR 2024) reuses the very same Sinkhorn solver (`otgan/sinkhorn.py`) to couple noise with data — no critic, no minimax: FID **11.4** in ~1.2 h.

This notebook is a thin narrative; training, evaluation and ablations live in the `otgan` package and its CLI.
More: [README](README.md) · [technical report](report/OT_GAN_report.pdf) · finance notebooks: [scenario reduction](examples/scenario_reduction.ipynb), [returns generation](examples/returns_generation.ipynb).

In [ ]:
# On Colab, uncomment to install the package (pulls torch/torchvision/torchmetrics):
# %pip install 'git+https://github.com/dan-allouche-qf/optimal-transport-gan.git'

# All outputs (data, weights, evaluations, logs) land under OT_GAN_ROOT
# (default: the current directory). On Colab, persist them to Drive:
# import os; os.environ["OT_GAN_ROOT"] = "/content/drive/MyDrive/OT_GAN"

## 1. Instant path — sample from the released weights

`otgan_mnist_sinkdiv_ema_v1.0.0.pt` (59.3 MB) is the EMA generator of the Sinkhorn-divergence run
(best epoch 7, FID 24.4 @10k) — weights + config only, exported with `scripts/export_generator.py`.
The CLI equivalent of the next two cells:

```bash
otgan sample --ckpt weights/otgan_mnist_sinkdiv_ema_v1.0.0.pt -n 64 -o samples.png
```

In [ ]:
import hashlib
import urllib.request
from pathlib import Path

CKPT_URL = (
    "https://github.com/dan-allouche-qf/optimal-transport-gan/"
    "releases/download/v1.0.0/otgan_mnist_sinkdiv_ema_v1.0.0.pt"
)
CKPT_SHA256 = "98d813494c0989fef900491416a78f304af2d2ea09e28b1611bf43d4254e5258"
ckpt_path = Path("weights/otgan_mnist_sinkdiv_ema_v1.0.0.pt")

# If the v1.0.0 release is not published yet, point this at any local export made
# with `python scripts/export_generator.py <full_ckpt.pt>` and re-run the cell.
local_checkpoint = None

if local_checkpoint is not None:
    ckpt_path = Path(local_checkpoint)
elif not ckpt_path.exists():
    try:
        ckpt_path.parent.mkdir(parents=True, exist_ok=True)
        print("downloading", CKPT_URL)
        urllib.request.urlretrieve(CKPT_URL, ckpt_path)
    except Exception as err:
        raise RuntimeError(
            "Could not download the released weights — the v1.0.0 release may not be "
            "published yet. Set `local_checkpoint` above to a generator export, or "
            "skip to the training demo below."
        ) from err

digest = hashlib.sha256(ckpt_path.read_bytes()).hexdigest()
if ckpt_path.name == "otgan_mnist_sinkdiv_ema_v1.0.0.pt" and digest != CKPT_SHA256:
    raise RuntimeError(f"SHA256 mismatch — corrupted download? got {digest}")
print(f"{ckpt_path} ({ckpt_path.stat().st_size / 1e6:.1f} MB) — sha256 {digest[:16]}... OK")

In [ ]:
import matplotlib.pyplot as plt
import torch
from torchvision.utils import make_grid

from otgan.config import Config
from otgan.models import OTGANGenerator
from otgan.trainer import denormalize

ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
cfg = Config(**ckpt["config"])  # every checkpoint carries its full training config
gen = OTGANGenerator(z_dim=cfg.z_dim, channels=cfg.channels)
gen.load_state_dict(ckpt["ema"])  # the EMA generator — the weights `otgan sample` uses
gen.eval()

torch.manual_seed(0)
with torch.no_grad():
    imgs = denormalize(gen(torch.randn(64, cfg.z_dim)))

plt.figure(figsize=(6, 6))
plt.imshow(make_grid(imgs, nrow=8).permute(1, 2, 0), cmap="gray")
plt.axis("off")
plt.title(f"Sinkhorn-divergence run, EMA generator (epoch {ckpt.get('epoch')}) — FID 24.4 @10k")
plt.show()

## 2. Training — a tiny demo of the better loss

Full runs belong to the CLI (`otgan train -c configs/mnist.yaml`; the 8-epoch runs in the results
table take ~3.5 h on Apple Silicon). The cell below runs a deliberately tiny budget with
`loss="sinkhorn_divergence"` — the Act-2 objective that took FID from 81.2 to 24.4 at equal budget.

In [ ]:
import dataclasses

from otgan import Config, OTGANTrainer

# Tiny on purpose: 2 epochs x 150 batches, 20 Sinkhorn iterations, no in-loop FID.
# (On Colab without a repo checkout, build the config directly: Config(dataset="mnist", ...).)
cfg = dataclasses.replace(
    Config.from_yaml("configs/mnist.yaml"),
    loss="sinkhorn_divergence",  # Act 2: debiased same-batch self-terms
    n_epochs=2,
    sinkhorn_iters=20,
    max_batches=150,
    fid_every=0,  # skip in-loop FID; evaluate afterwards with `otgan eval`
)
trainer = OTGANTrainer(cfg)
history = trainer.fit()

# Full-budget equivalent:
#   otgan train -c configs/mnist.yaml --override loss=sinkhorn_divergence

## 3. The evaluation harness

Every number in this repo comes from one harness (`otgan/metrics.py`):

- **FID @ 10k** — torchmetrics InceptionV3, `n_eval=10000` per side (the MNIST test set caps the
  real side at 10k). Small-sample FID is biased (Chong & Forsyth, CVPR 2020), so the legacy
  `n_eval=2048` numbers below are labeled as such.
- **KID** — the unbiased kernel estimator (Binkowski et al., ICLR 2018), reported ×1e3.
- **FID-LeNet** — the same Fréchet formula in the feature space of a MNIST-trained LeNet
  (`otgan/lenet.py`): Inception features are unreliable on 1-channel digits — our DCGAN baseline
  scores FID 72.3 but FID-LeNet 36.8, i.e. the two metrics *disagree*.
- **Floor** — the FID between the MNIST train and test sets is 1.45 (Inception) / 1.76 (LeNet);
  nothing below that is meaningful.
- **IS** is kept only as a sanity metric: it reads 2.33 vs 2.34 on corrected vs buggy OT-GAN —
  blind to a 107-point FID gap.

In [ ]:
# Evaluate any checkpoint with the shared harness (config is recovered from the file):
#   !otgan eval --ckpt weights/otgan_mnist_sinkdiv_ema_v1.0.0.pt
# -> {"fid": ..., "kid_mean": ..., "kid_std": ..., "fid_lenet": ..., "is_mean": ..., "is_std": ...}
#
# The released OT-CFM checkpoint goes through the same commands (the CFM trainer is
# selected from the config stored in the file):
#   !otgan sample --ckpt weights/otcfm_mnist_v1.0.0.pt -n 64 -o cfm_samples.png
#   !otgan eval   --ckpt weights/otcfm_mnist_v1.0.0.pt
#
# Resolution floor — FID between the MNIST train and test sets themselves:
#   !otgan eval -c configs/mnist.yaml --floor
# -> {"fid_floor": {"fid": 1.45, "fid_lenet": 1.76}}

## 4. Results — one harness across the three acts

![Energy distance vs Sinkhorn divergence](assets/loss_comparison.png)

*Act 2 at equal budget (8 epochs, same seed): the debiased Sinkhorn divergence reaches FID 24.4 vs
81.2 for the energy distance — and the critic's `D²` holds at ~0.077 instead of collapsing to ~1e-4.
The Sinkhorn-divergence run was still improving when the budget ended.*

Headline table (source of truth: [`assets/headline_results.md`](assets/headline_results.md)) —
FID/KID @ 10k samples, torchmetrics-Inception + MNIST-LeNet:

| model | budget | FID@10k | KIDx1e3 | FID-LeNet | IS |
|---|---|---|---|---|---|
| OT-GAN, buggy critic sign (final, 18 ep) | ~6 h MPS | 177.9 | 191.0 | 208.8 | 2.34 |
| OT-GAN, corrected (final, 18 ep) | ~6 h MPS | 70.8 | 65.7 | 77.0 | 2.33 |
| OT-GAN, energy distance (8 ep) | ~3.5 h MPS | 81.2 | 75.3 | 69.1 | 2.33 |
| OT-GAN, Sinkhorn divergence (8 ep) | ~3.5 h MPS | 24.4 | 13.8 | 60.1 | 1.98 |
| DCGAN baseline (10 ep) | ~15 min MPS | 72.3 | 74.6 | 36.8 | 2.19 |
| I-CFM, no coupling (30 ep) | ~1.2 h MPS | 10.9 | 8.9 | 12.8 | 1.91 |
| OT-CFM, Sinkhorn coupling (30 ep) | ~1.2 h MPS | 11.4 | 9.6 | 12.6 | 1.90 |
| *train-vs-test floor* | - | 1.45 | - | 1.76 | - |

OT-CFM controls:

- I-CFM control — independent coupling (`--override cfm_coupling=none`), same capacity and budget: FID 10.9 at NFE 100, 13.3 at NFE 10 — a wash vs OT-CFM at this scale (an honest null result; minibatch plans over 256 MNIST images approximate the global OT map weakly).
- NFE sweep — Euler steps vs FID at sampling time: OT-CFM 13.5 / 10.8 / 11.1 (FID-LeNet 34.6 / 15.2 / 13.6) at 10 / 50 / 100 Euler steps — ten-step sampling is cheap on Inception FID (≈ +2.7 FID vs the 50-step sweet spot) but FID-LeNet degrades ~2.5× (34.6 vs 13.6).

For honest context: Lucic et al. (NeurIPS 2018) report WGAN ≈ 6.7 / WGAN-GP ≈ 20.3 on MNIST at much
larger budgets. Nothing here is a SOTA claim — these are controlled comparisons under one harness at
laptop budgets.

## 5. Act 1 in detail — the five bugs and the critic-sign ablation

The five corrections, all locked by tests (table ported from the [README](README.md)):

| # | severity | original bug | fix |
|---|---|---|---|
| 1 | critical | **Critic trained in the wrong direction**: a single loss was *descended* by both optimizers, while the critic must **maximize** the energy distance | Critic takes a gradient **ascent** step (`(-D²).backward()`), generator descends; toggleable via `critic_sign` (the ablation below) |
| 2 | critical | Hard-coded `.cuda()` — crashed anywhere but CUDA (Mac/CPU) | `resolve_device()` (CUDA → MPS → CPU); tensors and models placed via `.to(device)` |
| 3 | major | **Scale mismatch**: generator `Tanh ∈ [-1, 1]` vs real images `∈ [0, 1]` — the critic could cheat on brightness | `Normalize((0.5,), (0.5,))` on the data; de-normalization for display |
| 4 | major | `DoubleBatchDataset(batch, batch)` → `X == X'`, so `E[W(X, X')] ≈ 0` | Two **independent** real minibatches per step (`2·B` images split in two) |
| 5 | major | **Sinkhorn in the exponential domain** (`exp(-C/ε)`) — overflow for small ε | **Log-domain** Sinkhorn (`logsumexp`); transport plan detached (envelope theorem) |

The worst one (#1) has its own ablation harness:

```bash
otgan ablate -c configs/mnist.yaml --axis critic_sign   # other axes: epsilon, g2c_ratio
```

The original 18-epoch ablation (legacy protocol, `n_eval=2048` — biased upward vs the @10k table):
`critic_sign=True` final FID **77.2** (best **69.9** at epoch 8) vs `critic_sign=False` **185.3** —
with IS identical at 2.32 for both, again useless. Raw curves:
`assets/history_critic_sign_{on,off}.csv`; table: [`assets/ablation_table.md`](assets/ablation_table.md).

## 6. Side chapter — the same engine on market data

`otgan/finance` reuses the same Sinkhorn solver on financial scenarios; executed notebooks live in
`examples/`:

- [scenario_reduction.ipynb](examples/scenario_reduction.ipynb) — OT scenario reduction with zero
  training: distortion 0.0088 vs 0.0120 for random subsampling (~37% higher for random), and a tie
  with k-means on held-out CVaR. CLI: `otgan finance-reduce -c configs/finance_returns.yaml -K 100 --compare`.
- [returns_generation.ipynb](examples/returns_generation.ipynb) — a 1D OT-GAN on GJR-GARCH returns
  (Glosten–Jagannathan–Runkle 1993): at a 5-minute budget every stylized-fact **sign** comes out
  right (incl. leverage correlation −0.29), with honest magnitude overshoots (kurtosis, |r|
  autocorrelation). The critic-sign ablation replays in minutes in 1D.
  CLI: `otgan finance-train -c configs/finance_returns.yaml`, then `otgan finance-eval -c ... --ckpt ...`.

Ground truth: [`assets/finance/reduction_table.md`](assets/finance/reduction_table.md),
[`assets/finance/stylized_facts.md`](assets/finance/stylized_facts.md).

## 7. Where this sits in 2026

GANs lost the generative crown to diffusion and flow models, but minimax training is not dead
(R3GAN — "The GAN is dead; long live the GAN!", NeurIPS 2024) — and optimal transport quietly moved
from the loss (OT-GAN, 2018) into the data coupling (OT-CFM, 2024), which is exactly the arc of
this repo. What carries over is the harness: controlled comparisons, domain-aware metrics, honest
floors — the same discipline a quant applies to a backtest.

**References.** Salimans, Zhang, Radford & Metaxas, ICLR 2018 (OT-GAN) · Cuturi, NeurIPS 2013
(Sinkhorn) · Genevay, Peyré & Cuturi, AISTATS 2018 · Feydy et al., AISTATS 2019 (debiased Sinkhorn
divergences) · Lipman et al., ICLR 2023 (flow matching) · Tong et al., TMLR 2024 (OT-CFM) ·
Binkowski et al., ICLR 2018 (KID) · Parmar et al., CVPR 2022 (FID resizing) · Chong & Forsyth,
CVPR 2020 (FID bias) · Lucic et al., NeurIPS 2018 · R3GAN, NeurIPS 2024 · Finance: Xu et al.,
NeurIPS 2020 (COT-GAN); Wiese et al., 2020 (Quant GANs); Cont et al. (Tail-GAN); Ni et al.,
ICAIF 2021 (Sig-WGAN); Glosten, Jagannathan & Runkle, 1993 (GJR-GARCH).

*Dan Allouche & Nicolas Dahan (ENSAE) — MIT license.*